In [32]:
!uv pip install  langchain-google-genai



Using Python 3.10.20 environment at: /Users/kbshal/mero_space/sarathi_academy/.venv
Checked 1 package in 28ms


In [54]:
import logging
import os

In [31]:
import ast
import operator

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def safe_eval(expression: str) -> float:
    """Safely evaluate a basic arithmetic expression."""
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression}")

    tree = ast.parse(expression, mode="eval")
    return _eval(tree.body)


In [46]:

def verified(tool_fn):
    """Wrap a tool so its output is validated before going back to the model."""
    def wrapper(*args, **kwargs):
        result = tool_fn.invoke(kwargs or args)
        if isinstance(result, float) and (result != result):   # NaN check
            raise ValueError("Tool returned NaN")
        logging.info(f"TOOL {tool_fn.name} -> {result}")
        return result
    return wrapper

In [33]:
def word_count(text): 
    return len(text.split())
    
def times_ten(n):     
    return int(n) * 10

TOOLS = {"word_count": word_count, "times_ten": times_ten}

# A real agent asks an LLM "what next?" each step. We hard-code 2 steps here.
PLAN = [("word_count", "the quick brown fox jumps over the lazy dog"),
        ("times_ten",  None)]        # None = use the last result

def run_agent():
    last = None
    for name, arg in PLAN:
        arg = last if arg is None else arg   # decide
        last = TOOLS[name](arg)                # act -> observe
        print(name, arg, "->", last)
    print("final answer:", last)

In [34]:
from langchain_core.tools import tool

@tool
def word_count(text: str) -> int:
    """Count the number of words in some text.
    
    input: str
    return: int
    
    """
    return len(text.split())

@tool
def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression, e.g. '9 * 10'."""
    return safe_eval(expression)      # never eval() model input

TOOLS = [word_count, calculator]

In [35]:
import os
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key=os.getenv("GOOGLE_API_KEY"),  # or set GOOGLE_API_KEY env var
)

agent = create_agent(model, tools=TOOLS)


In [43]:
GOAL = "How many words in 'the quick brown fox jumps over the lazy dog', and that count times 10?"

In [63]:
try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": GOAL}]},
        config={"recursion_limit": 10},          # guard #1: cap the loop
    )
    print(result["messages"][-1].content)
except Exception as e:
    logging.error(f"Agent failed: {e}")          # guard #4: log failures loudly
    print("Sorry, I couldn't complete that task.")

[{'type': 'text', 'text': 'The phrase "the quick brown fox jumps over the lazy dog" contains **9** words.\n\nMultiplying that count by 10 gives **90** (9 × 10).', 'extras': {'signature': 'EvAFCu0FARFNMg9QT2SmziKNQSSC2OqAFqnNU93qv4/OLSCpRIbtZVjdgbj4b+VS/t6jpFNIeZPO6jEQOJ+RSMd30pHEw+TWkaNVazioRRNkhL55tbcEcYUKWp71IywxsB1D5I2oE0ZTfYpNgNiRuFMi2nJUiQIRr+x02iXohxWKHIfAf/Qqb0/tjq1768Nxmk53fPeBPbWHmX5Fdw912AjeX89XSDyhKRggMqqfWPRopIrs5PzL73WeWDub9uN0YlqunyiH4YEk/OUF7fFyfnlJINtt+tjiY2GTpWRcoFWC/DQi3k82OCwO2oFOLSbWiXlHdS6nU8LcNx2GCIczAtYidw3W5ceNNOlTlkW6EavFEz95pOVo7kRnQ7hQIXEPt7YeC+rsF3VOaDVURdohdahan0V7pyvSPO9j6ZJ4O8jos+Rye0Fzb7QQWZp0a/4OD9WIqFW8AErMKTxCav6/zjHLxdaYf4xRCBkvIbXqxDOJizyTlYwl3W6e+6vFehS83aXzkisv/1dW6o5Xb/Zyew4NpztG3FnWVZB6//qrSXPon5kx+0AE9LNSrl1K2lfi5Zdu1KYRYOi4pq4HNq0Py1taQFBzRg+BnIyHw6oImO4iVz+9uwBDt1dWCFDXOKiBggjuuz6hf4+WzRXu8sATNXzM+7FFMIQCeFAhApcXqMidisZHaGSJAQFODzWviYURe56GY5Ysgo9xB9xME4djmNaIqdLlPY4eDMvFQHEJ23uNcuCZL9xqjUz7mjPQNNgL7aRafB59Szeh3zjGhzLMLbxlVJnprqUGZAU5dCtHULeG

In [64]:
@tool
def delete_file(path: str) -> str:
    """Delete a file. Requires human confirmation."""
    answer = input(f"Approve deleting '{path}'? [y/N] ")
    if answer.lower() != "y":
        return "Denied by human."
    os.remove(path)
    return f"Deleted {path}"


In [66]:
NEW_TOOLS = [delete_file]

In [67]:
agent = create_agent(model, tools=NEW_TOOLS)

In [68]:
DEL_GOAL = "delete this file, test_file_delete.txt"

try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": DEL_GOAL}]},
        config={"recursion_limit": 10},          # guard #1: cap the loop
    )
    print(result["messages"][-1].content)
except Exception as e:
    logging.error(f"Agent failed: {e}")          # guard #4: log failures loudly
    print("Sorry, I couldn't complete that task.")

Approve deleting 'test_file_delete.txt'? [y/N]  y


[{'type': 'text', 'text': 'The file `test_file_delete.txt` has been successfully deleted.', 'extras': {'signature': 'Eq4BCqsBARFNMg838EaM9kPiDz4zFQhjn9X1Rtl/886nei3sq1lVL7LnIrMBErKhm6okP+HUK9/XHoPSqzoxFg9H/Ta3EKl7qLOPYuURuwqwNxejKH1ewRFZ0h89mAoYRBtOeKyekDZTxk3BiQz1vFHRh/oQHg2TwNPT5xMJqGNsN+A8nQ0jmn7ruekDN4xD/UQkFIizm3Dckozd0yM+KWNqjj9fR+Q2y0KZ9Bhthgic'}}]


In [60]:
import os

In [62]:
x = 10
y = 5
user_input = input("pls enter your input: ")
# Evaluate a math string using active variables
result = eval(user_input)
print(result)  # Output: 20

pls enter your input:  os.system('ls')


__pycache__
attention.ipynb
course_notes.py
data_gen.py
deep_neural_networks.ipynb
embeddings.ipynb
fe1_temp.json
feature_engineering-(1).ipynb
feature_engineering.ipynb
fri_jul_24.ipynb
full_scale_pipeline.ipynb
handwritten_notebook.ipynb
instagram_clustering.ipynb
k_means.ipynb
learning_sql.ipynb
ml_fundamentals.ipynb
pytorch_day_1.ipynb
pytorch_day_2.ipynb
rag_again_full_pipeline.ipynb
rag_full.ipynb
rag_full_pipeline_llm_call.ipynb
ragas.ipynb
random_forest.ipynb
requirements.txt
review_data.py
river_data_analysis.ipynb
sarathi_academy_course_aiml.pdf
school.db
testfile_delete.txt
tokenizer.ipynb
tool_calling_agent.ipynb
training_deep_neural_networks.ipynb
tue_21_jul.ipynb
vector.ipynb
0
